# ECFP4 Encoder

made this for tuning and evaluating the fingerprint MLP encoder on BACE-1 pIC50 regression. the actual encoder code goes in `ai/src`.

NOTE: i trained it here w/ a throwaway linear head just to gauge how good its features are. the real heads and the projection to the shared dimensions are gonna be implemented later. also the murcko scaffolding for scaffold split have yet to be implemented (at the time of writing this), so just making one up here

In [ ]:
import sys

sys.path.insert(0, "../src")

# hyperparams
N_BITS = 2048  # 1024 or 2048
HIDDEN_DIMS = (512, 256)
DROPOUT = 0.6
LR = 1e-3
WEIGHT_DECAY = 1e-3
BATCH_SIZE = 64
EPOCHS = 100
SEED = 314159265359

In [ ]:
import copy
from datetime import datetime
from pathlib import Path

import bioactivity_loader as bl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from featurize.fingerprint import mol_to_ecfp4
from fingerprint_encoder import FingerprintEncoder
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import r2_score, roc_auc_score, root_mean_squared_error
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Data

In [ ]:
df = bl.load_bace1()
print(f"{len(df)} molecules, {df['active'].mean():.1%} active")
df.head()

ECFP4 fingerprints come from `mol_to_ecfp4` in `ai/src/featurize/fingerprint.py`

In [ ]:
mols = df["smiles"].map(Chem.MolFromSmiles)
X = torch.from_numpy(np.stack([mol_to_ecfp4(m, N_BITS) for m in mols]))
y = torch.tensor(df["pIC50"].to_numpy(), dtype=torch.float32)
X.shape, X.dtype

### Scaffold split (80/10/10)

mols grouped by Bemis-Murcko scaffold and whole groups go to one split. largest groups first (so train gets common scaffolds and val/test get the rarer ones)

In [ ]:
scaffolds = df["smiles"].map(lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s))
groups = sorted(
    df.groupby(scaffolds).indices.values(), key=lambda idx: (-len(idx), idx[0])
)

n = len(df)
train_idx, val_idx, test_idx = [], [], []
for idx in groups:
    if len(train_idx) + len(idx) <= 0.8 * n:
        train_idx += idx.tolist()
    elif len(val_idx) + len(idx) <= 0.1 * n:
        val_idx += idx.tolist()
    else:
        test_idx += idx.tolist()

print(f"train {len(train_idx)}, val {len(val_idx)}, test {len(test_idx)}")

## Model

there may be some pretrained alternatives to A/B with, but it's kinda tough finding a pretrained ECFP-MLP encoder specifically. but if I can find any I'll try to compare performance with ours. so far though **CLAMP** looks semi-promising. leaving it here in case we use it later

- **CLAMP** (`ml-jku/clamp`): a shallow MLP over ECFP + RDKit + MACCS fingerprints thats pretrained contrastively on ChEMBL bioactivity assays. it's the closest match, but input isn't just ECFP4, and ChEMBL pretraining could leak BACE-1 labels into a ChEMBL derived test set.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"training on {DEVICE}")

torch.manual_seed(SEED)
encoder = FingerprintEncoder(N_BITS, HIDDEN_DIMS, DROPOUT)
# TODO(pretrained): swap in a pretrained encoder here to A/B with ours
head = nn.Linear(encoder.out_dim, 1)  # fake head
nn.init.constant_(head.bias, y[train_idx].mean().item())  # start at train mean
model = nn.Sequential(encoder, head).to(DEVICE)


@torch.no_grad()
def predict(idx):
    model.eval()
    return model(X[idx].to(DEVICE)).squeeze(-1).cpu()


def rmse(idx):
    return root_mean_squared_error(y[idx], predict(idx))


baseline = root_mean_squared_error(
    y[val_idx], np.full(len(val_idx), y[train_idx].mean().item())
)
print(f"val RMSE of always predicting the train mean: {baseline:.3f}")

## Train

In [ ]:
loader = DataLoader(
    TensorDataset(X[train_idx], y[train_idx]),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_val, best_state = float("inf"), None
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(xb).squeeze(-1), yb)
        loss.backward()
        optimizer.step()

    train_rmse, val_rmse = rmse(train_idx), rmse(val_idx)
    history.append((train_rmse, val_rmse))
    if val_rmse < best_val:
        best_val, best_state = val_rmse, copy.deepcopy(model.state_dict())
    print(f"epoch {epoch + 1:3d}  train RMSE {train_rmse:.3f}  val RMSE {val_rmse:.3f}")

## Evaluate

best val weights, scored once on test. ROC-AUC uses predicted pIC50 as the score against the `active` label (pIC50 >= 6); we don't have a classification head yet

In [ ]:
model.load_state_dict(best_state)

metrics = {}
for name, idx in [("val", val_idx), ("test", test_idx)]:
    pred = predict(idx)
    metrics[name] = {
        "RMSE": root_mean_squared_error(y[idx], pred),
        "R2": r2_score(y[idx], pred),
        "ROC-AUC": roc_auc_score(df["active"].iloc[idx], pred),
    }
    print(f"{name:>4}: " + "  ".join(f"{k} {v:.3f}" for k, v in metrics[name].items()))

In [ ]:
train_curve, val_curve = zip(*history)
epochs = range(1, len(history) + 1)
pred = predict(test_idx)
lims = [y.min().item(), y.max().item()]

fig, (ax_curve, ax_parity) = plt.subplots(1, 2, figsize=(12, 5))

ax_curve.plot(epochs, train_curve, label="train")
ax_curve.plot(epochs, val_curve, label="val")
ax_curve.axhline(baseline, color="gray", linestyle="--", label="predict train mean")
ax_curve.set_xlabel("epoch")
ax_curve.set_ylabel("RMSE (pIC50)")
ax_curve.set_title("learning curve")
ax_curve.legend()

ax_parity.scatter(y[test_idx], pred, s=8, alpha=0.6)
ax_parity.plot(lims, lims, color="gray", linestyle="--")
ax_parity.set_xlabel("actual pIC50")
ax_parity.set_ylabel("predicted pIC50")
ax_parity.set_title("test set")
table = f"{'':>4}  {'RMSE':>5}  {'R2':>6}  {'ROC-AUC':>7}\n" + "\n".join(
    f"{name:>4}  {m['RMSE']:5.3f}  {m['R2']:6.3f}  {m['ROC-AUC']:7.3f}"
    for name, m in metrics.items()
)
ax_parity.text(
    0.03,
    0.97,
    table,
    transform=ax_parity.transAxes,
    va="top",
    family="monospace",
    fontsize=9,
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
)

fig.suptitle(
    f"N_BITS={N_BITS}  HIDDEN_DIMS={HIDDEN_DIMS}  DROPOUT={DROPOUT}  LR={LR}  "
    f"WEIGHT_DECAY={WEIGHT_DECAY}  BATCH_SIZE={BATCH_SIZE}  EPOCHS={EPOCHS}  SEED={SEED}",
    fontsize=10,
)

fig.tight_layout()
out_dir = Path("../outputs/encoders/ECFP4")
out_dir.mkdir(parents=True, exist_ok=True)
# keep the last run's figure as -prev for comparison
out_path = out_dir / "ECFP4-encoder.png"
if out_path.exists():
    out_path.replace(out_dir / "ECFP4-encoder-prev.png")
fig.savefig(out_path, dpi=150)
plt.show()

## Run log

every run appends one row to `outputs/encoders/ECFP4/runs.csv`. pick the best config by `val_RMSE`, not test. re-running this cell alone logs the same run twice

In [ ]:
val_curve = [val for _, val in history]
row = {
    "time": datetime.now().isoformat(timespec="seconds"),
    "N_BITS": N_BITS,
    "HIDDEN_DIMS": HIDDEN_DIMS,
    "DROPOUT": DROPOUT,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "SEED": SEED,
    "best_epoch": int(np.argmin(val_curve)) + 1,  # epoch of the checkpoint used above
    **{f"{split}_{k}": v for split, m in metrics.items() for k, v in m.items()},
}

log_path = out_dir / "runs.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not log_path.exists(), index=False)

runs = pd.read_csv(log_path)
runs.sort_values("val_RMSE").head(10)